# M6 · Lab 3 — Great Expectations Data Validation
### Module 6 — Monitoring, Testing & Drift Detection | Spine Project: Truck Delay Classification

| | |
|---|---|
| **Duration** | 60 min |
| **Difficulty** | Intermediate |
| **Tools** | Python 3.12.10, `great-expectations>=0.18,<1.0`, pandas |
| **AWS** | None (local notebook — Lab 4 wires GE's pass/fail into SNS) |
| **Prerequisite** | Lab 2 (same `data/` folder with the real M3 reference frame) |
| **Builds toward** | Lab 4 + Lab 5 (combined pipeline runs GE → Evidently → SNS) |
| **Cost** | ₹0 — all local |

---
Lab 2 caught **statistical** drift. This lab catches **schema** breaks — corrupted values, NULL spikes, type changes,
out-of-range numbers. Drift is gradual and probabilistic; schema breaks are sudden and binary. Production needs **both**.

## 💻 Where to run this — SageMaker (recommended in class), Colab, or local

**In class: the SageMaker Notebook Instance your instructor provisioned** (`m6-truck-delay-monitoring`). Everything is
pre-installed (`evidently`, `great-expectations`, `xgboost`) and the real `data/` folder is already there — just open this
notebook and run.
- **Kernel:** `conda_python3`
- **Data:** `data/reference/final_features.csv` + `data/artifacts/` ship with the labs on the instance (the `M6_DATA_DIR` default is `data`).
- **Auto-stop:** the instance stops itself after ~45 min idle to save cost — your work, installed packages, and saved
  artifacts persist on the EBS volume. Just hit **Start** the next day; **M7 and M8 reuse this same instance.**
- **AWS access (Labs 4/5):** the instance's IAM role already grants `sns:Publish`, so there are **no access keys to configure.**

**Portable fallback (self-paced / post-course):** this notebook also runs on **Google Colab** or **local Jupyter** — the
setup cell below detects the environment and pip-installs what's missing. On Colab, upload the `data/` folder (or set
`M6_DATA_DIR`). Locally, use **Python 3.12.10**. The data + model are the *real* M3 artifacts either way — nothing synthetic.

## Learning Objectives
1. Explain what **Great Expectations (GE)** solves and where the GE/Evidently line sits.
2. Auto-profile the **real** reference frame into a starting **expectation suite**.
3. Hand-edit the suite to add domain rules the profiler can't infer.
4. Validate a **clean** batch (100% pass) and a **corrupted** batch (specific row-level failures).
5. Emit a structured failure payload + generate **Data Docs** (the team's "what must the data look like?" contract).

## Business Context
A month into production, the upstream scheduling service started emitting `route_id` as the string `"R-04522"` instead
of the integer `4522`. The Truck Delay app didn't crash — it coerced the bad value to NaN, produced a low-confidence
prediction, and operations treated it as "on time". **Three days of misrouted dispatches** before anyone joined the dots.

That's **not drift** — a column of NaNs has no distribution to compare. It's a **schema break**, and it needs *declarative
validation*: you write down "`truck_age` is an integer in [1, 30]" and every incoming batch is checked against the
contract. One violating value fails the batch — loudly, with a row-level message — before the bad data reaches the model.

## Prerequisites — the **real** M3 reference frame (no synthetic data)
Same `data/` folder as Lab 2. We profile the genuine training distribution — **`data/reference/final_features.csv`**
(12,308 × 37, regenerated from M3 Lab B and round-trip-validated against the model). The *only* synthetic thing here is
the deliberately **corrupted batch** in Step 7 — you can't demonstrate catching corruption without some.

> **Why pin GE `<1.0`?** GE v1.0 (Sep 2024) is a breaking rewrite. The course — and the Lab 5 production pipeline that
> consumes this suite — uses the stable **0.18** line. Once you know 0.18, migrating to v1 is a half-day.

In [ ]:
# ── Environment setup (Colab or local) ────────────────────────────────────────
import sys, subprocess, os
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
try:
    import great_expectations as gx
    print("great_expectations:", gx.__version__)
except ImportError:
    print("Installing great-expectations 0.18 …")
    _pip("great-expectations>=0.18,<1.0", "pandas==2.2.0", "numpy<2.0")
    import great_expectations as gx
    print("great_expectations:", gx.__version__)

In [ ]:
# ── Imports + load the real reference frame ───────────────────────────────────
import json
import numpy as np
import pandas as pd

DATA_DIR = os.environ.get("M6_DATA_DIR", "data")
REF_CSV  = os.path.join(DATA_DIR, "reference", "final_features.csv")
assert os.path.exists(REF_CSV), f"Real reference frame missing at {REF_CSV} (copy Module 6/labs/data/ next to this notebook)."

ref = pd.read_csv(REF_CSV)
print(f"Reference frame: {ref.shape}  (expected 12308 x 37)")
ref.head(3)

## Step 1 · Concepts — what's an "Expectation"?
A single **expectation** is one assertion about a column:
```python
expect_column_values_to_be_between("truck_age", min_value=1, max_value=30)
```
There are ~50 built-ins:

| Category | Example | Checks |
|---|---|---|
| Schema | `expect_column_to_exist` | the column is present |
| Type | `expect_column_values_to_be_of_type` | int / float / str |
| Range | `expect_column_values_to_be_between` | min/max bounds |
| Set | `expect_column_values_to_be_in_set` | value ∈ enum |
| Null | `expect_column_values_to_not_be_null` | NULL rate (`mostly=0.99`) |
| Cardinality | `expect_column_distinct_values_to_be_in_set` | exact allowed set |

An **expectation suite** is a JSON file of dozens of these. A **checkpoint** runs a suite against new data.
Flow: **profile → edit → validate → publish**.

## Step 2 · Create a file-backed GE project + register the reference as a runtime batch

In [ ]:
from great_expectations.data_context import FileDataContext
from great_expectations.core.batch import RuntimeBatchRequest

# Creates ./great_expectations/ (committable) the first time; loads it thereafter.
try:
    context = FileDataContext.create(project_root_dir=".")
except Exception:
    context = gx.get_context(context_root_dir="./great_expectations")
print("GE project root:", context.root_directory)

# A runtime datasource lets us validate in-memory pandas DataFrames (the production pattern).
if "truck_delay_runtime" not in [ds["name"] for ds in context.list_datasources()]:
    context.add_datasource(
        name="truck_delay_runtime",
        class_name="Datasource",
        execution_engine={"class_name": "PandasExecutionEngine"},
        data_connectors={
            "default_runtime_data_connector_name": {
                "class_name": "RuntimeDataConnector",
                "batch_identifiers": ["default_identifier_name"],
            }
        },
    )
print("Datasources:", [ds["name"] for ds in context.list_datasources()])

## Step 3 · Auto-profile the reference frame into a starting suite

In [ ]:
from great_expectations.profile.user_configurable_profiler import UserConfigurableProfiler

SUITE_NAME = "truck_delay_features"
context.add_or_update_expectation_suite(expectation_suite_name=SUITE_NAME)

def batch_request_for(df, label):
    '''Wrap an in-memory DataFrame as a GE RuntimeBatchRequest.'''
    return RuntimeBatchRequest(
        datasource_name="truck_delay_runtime",
        data_connector_name="default_runtime_data_connector_name",
        data_asset_name="truck_delay",
        runtime_parameters={"batch_data": df},
        batch_identifiers={"default_identifier_name": label},
    )

validator = context.get_validator(
    batch_request=batch_request_for(ref, "initial_profile"),
    expectation_suite_name=SUITE_NAME,
)

profiler = UserConfigurableProfiler(
    profile_dataset=validator,
    excluded_expectations=["expect_column_quantile_values_to_be_between"],  # noisy
    not_null_only=True,
)
suite = profiler.build_suite()
context.save_expectation_suite(suite, expectation_suite_name=SUITE_NAME)
print(f"Profiled {len(suite.expectations)} expectations across {ref.shape[1]} columns")

## Step 4 · Hand-edit — add the domain rules the profiler can't infer
The profiler writes *empirical* bounds (e.g. `truck_age ∈ [1, 23]` because that's what it saw). We widen them to the
*business* range so next year's 24-year-old truck doesn't trip a false alarm — and we add cross-column + enum rules.

In [ ]:
v = context.get_validator(
    batch_request=batch_request_for(ref, "domain_rules"),
    expectation_suite_name=SUITE_NAME,
)

# Business range (not the empirical one the profiler inferred)
v.expect_column_values_to_be_between("truck_age", min_value=1, max_value=30, mostly=1.0)
v.expect_column_values_to_be_between("age", min_value=18, max_value=75, mostly=1.0)
v.expect_column_values_to_be_between("experience", min_value=0, max_value=60, mostly=1.0)

# Target must be binary
v.expect_column_distinct_values_to_be_in_set("delay", value_set=[0, 1])

# Categorical enums (note: 'Unknown' is a legal NaN-fill value from M3 Lab B)
v.expect_column_values_to_be_in_set("fuel_type", value_set=["diesel", "gas", "Unknown"])
v.expect_column_values_to_be_in_set("gender", value_set=["male", "female", "Unknown"])
v.expect_column_values_to_be_in_set("driving_style",
                                    value_set=["proactive", "conservative", "aggressive", "Unknown"])
v.expect_column_values_to_be_in_set("is_midnight", value_set=[0, 1])
v.expect_column_values_to_be_in_set("accident", value_set=[0, 1])

# Cross-column domain rule: a driver's experience can't exceed their age
v.expect_column_pair_values_a_to_be_greater_than_b(column_A="age", column_B="experience", or_equal=True)

v.save_expectation_suite(discard_failed_expectations=False)
suite = context.get_expectation_suite(SUITE_NAME)
print(f"Suite now has {len(suite.expectations)} expectations (profiled + hand-added domain rules)")

## Step 5 · Validate a CLEAN batch — the reference itself should pass ~100%

In [ ]:
res_clean = context.get_validator(
    batch_request=batch_request_for(ref, "clean_validation"),
    expectation_suite_name=SUITE_NAME,
).validate()

print(f"Success:      {res_clean['success']}")
print(f"Expectations: {res_clean['statistics']['successful_expectations']}"
      f" / {res_clean['statistics']['evaluated_expectations']}")
print(f"Success %:    {res_clean['statistics']['success_percent']:.1f}")
# If <100%, a hand-edit over-tightened — loosen the offending range and re-run Step 4.

## Step 6 · Build a deliberately CORRUPTED batch (the only synthetic part)

In [ ]:
def make_corrupt_batch(reference: pd.DataFrame, n: int = 500, seed: int = 7) -> pd.DataFrame:
    '''Sample real rows and inject four classes of corruption GE should catch.'''
    b = reference.sample(n=n, random_state=seed).reset_index(drop=True).copy()
    b.loc[b.index[:25], "truck_age"] = -3                         # 1) out-of-range (negative)
    null_idx = b.sample(frac=0.05, random_state=1).index
    b.loc[null_idx, "ratings"] = np.nan                           # 2) NULL spike
    b.loc[b.index[300:310], "fuel_type"] = "hydrogen"             # 3) unknown category
    b.loc[b.index[400:402], "delay"] = 2                          # 4) target out of {0,1}
    return b

corrupt = make_corrupt_batch(ref)
print(f"Corrupted batch: {corrupt.shape}  (negative truck_age, null ratings, 'hydrogen' fuel, delay=2)")

## Step 7 · Validate the corrupted batch — GE should fail with specific row-level errors

In [ ]:
res_bad = context.get_validator(
    batch_request=batch_request_for(corrupt, "corrupted_validation"),
    expectation_suite_name=SUITE_NAME,
).validate()

print(f"Success: {res_bad['success']}")
print(f"Failed:  {res_bad['statistics']['unsuccessful_expectations']}"
      f" / {res_bad['statistics']['evaluated_expectations']}\n")
print("Failures:")
for r in res_bad["results"]:
    if not r["success"]:
        e = r["expectation_config"]
        col = e["kwargs"].get("column", e["kwargs"].get("column_A", "<table>"))
        unexpected = r["result"].get("unexpected_count", "?")
        sample = r["result"].get("partial_unexpected_list", [])[:3]
        print(f"  X {e['expectation_type']:42s} col={col:14s} unexpected={unexpected}  sample={sample}")

## Step 8 · The structured failure payload (what Lab 4 / Lab 5 publish to SNS)

In [ ]:
def ge_failure_payload(results) -> dict:
    '''Reduce a GE validation result to the compact dict the alerting pipeline sends.'''
    return {
        "success": bool(results["success"]),
        "evaluated": results["statistics"]["evaluated_expectations"],
        "passed": results["statistics"]["successful_expectations"],
        "failed": results["statistics"]["unsuccessful_expectations"],
        "success_percent": round(results["statistics"]["success_percent"], 1),
        "failures": [
            {
                "expectation": r["expectation_config"]["expectation_type"],
                "column": r["expectation_config"]["kwargs"].get("column"),
                "unexpected_count": r["result"].get("unexpected_count", 0),
                "sample": r["result"].get("partial_unexpected_list", [])[:3],
            }
            for r in results["results"] if not r["success"]
        ],
    }

import pprint; pprint.pp(ge_failure_payload(res_bad))

## Step 9 · Generate Data Docs — the team's HTML schema contract

In [ ]:
context.build_data_docs()
docs = context.get_docs_sites_urls()
print("Data Docs built:")
for d in docs:
    print("  ", d["site_url"])
# Open that file:// URL in a browser. Expectations tab = the contract; Validation Results tab = run history.

## Step 10 · Artifact chain — what downstream labs consume
The committed output is **`great_expectations/expectations/truck_delay_features.json`** plus the project around it.
**Lab 4** (combined exploration) and **Lab 5** (production script) load this exact suite by name and run it against each
incoming batch *before* the Evidently drift check (GE is cheap → fail fast on schema).

In [ ]:
suite_path = os.path.join("great_expectations", "expectations", f"{SUITE_NAME}.json")
print("Suite persisted at:", suite_path, "->", os.path.exists(suite_path))
print("Pass this folder to Lab 4 / Lab 5 (they call context.get_validator(..., expectation_suite_name='truck_delay_features')).")

## GE vs Evidently — where to draw the line
| | **Great Expectations** (this lab) | **Evidently** (Lab 2) |
|---|---|---|
| Question | "does this batch satisfy the schema contract?" | "is the distribution shifting over time?" |
| Fires when | a *single* row violates a rule | statistical distance crosses a threshold |
| Best for | schema breaks, type changes, NULL spikes, out-of-range | slow drifts, monsoon precip shift, fleet aging |
| Misses | subtle statistical shifts | an all-NULL column (degenerate distribution) |
| Cost | cheap (iterate rules) | ~1 s (Wasserstein on 30 cols × 1k rows) |

**Rule of thumb:** if it would *break your code* → GE catches it. If it would *degrade your accuracy* → Evidently catches it.

## Verification Checklist
- [ ] `great_expectations/` project created; `expectations/truck_delay_features.json` exists
- [ ] Clean validation (reference) reports ~100% pass
- [ ] Corrupted validation fails, naming `truck_age`, `ratings`, `fuel_type`, `delay`
- [ ] `ge_failure_payload(...)` returns the compact failure dict
- [ ] Data Docs HTML opens and shows the suite + both validation runs
- [ ] You can state the GE-vs-Evidently difference in two sentences

## What's next — Lab 4
You now have two independent monitors: an Evidently drift detector (Lab 2) and a GE validator (Lab 3). **Lab 4** fuses
them into one flow — GE first (fail fast), then Evidently, then publish a structured alert to the **SNS topic from Lab 1**
— and returns an exit code so it composes with cron / Airflow / EventBridge. **Lab 5** ships that flow as a production `.py`.

## Troubleshooting
| Symptom | Fix |
|---|---|
| `ImportError: UserConfigurableProfiler` | GE v1 installed — pin `great-expectations>=0.18,<1.0` |
| `len(suite.expectations) == 0` | the validator wasn't fed a real batch — confirm `runtime_parameters={"batch_data": ref}` |
| "no datasource named ..." on validate | re-run Step 2 (the runtime datasource must be registered in this kernel) |
| clean validation < 100% | a hand-edit over-tightened — widen the offending range in Step 4 |
| `FileDataContext.create` errors "already exists" | fine — the `except` branch loads the existing project |
| Data Docs "no validations to render" | run a validation (Step 5) before `build_data_docs()` |